# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanKh4lil/ML-WEEK-1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Provisional lane: Lane 4 — CTR / Engagement Opportunity Scoring.**

I'm picking this lane because the starter data shows a strong, checkable position-vs-CTR relationship (CTR should fall as average position gets worse), which means I can define "underperforming for its position" honestly instead of guessing. It also has a clear, boring, high-leverage fix: most CTR problems are addressed with a title/meta rewrite, which is cheap and fast compared to a full content refresh. That makes the lane a good fit for a 7-week internship — I can build a defensible ranking without needing the full 79M-row warehouse to get started, and I can extend to the warehouse's `fact_content_daily_performance` later if I want more history or a future-window label. I'm keeping Lane 2 (Refresh Scoring) as a backup, since the two lanes share most of the same features and I can pivot without losing work.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Unit of analysis:** one content item (page), scored using its trailing-90-day metrics.

**The question:** Among pages that are already visible in search (they get real impressions at a reasonable average position), which ones are capturing fewer clicks than similar, similarly-ranked pages — and so deserve a title/meta/snippet review first?

**The decision it improves:** which pages an SEO editor with limited time reviews first for a CTR-focused fix (title, meta description, snippet structure), instead of picking pages by gut feel or by raw impression volume alone.

**Who acts, and how:** an SEO editor or content strategist. They'd take my ranked list, open the top N pages, and rewrite the title/meta or restructure the snippet — not touch page content or run a full rewrite.

**Cost of a wrong call:**
- *False positive* (page flagged as an opportunity but CTR is actually fine, e.g. it's a branded/informational query where low CTR is normal): wastes 15-30 minutes of editor time per page on a rewrite that won't move the needle, and risks a title change that displaces a query mix that was working.
- *False negative* (a genuinely underperforming page never surfaces): a page keeps leaking clicks it could otherwise capture, with the cost compounding every day it sits unreviewed.
- Because the action is reversible and cheap (a title edit, not a content rewrite), the cost of being wrong is moderate, not severe — this is a decision-support ranking, not a high-stakes automated action.

**Why data/ML helps rather than a fixed if-statement:** "low CTR" only means something *relative to position* — a 2% CTR is bad at position 2 but great at position 12 — and the position-to-expected-CTR curve isn't a straight line (see numbers below), so a single hard-coded threshold either flags too many page-1 items or misses striking-distance items. A model or a tier-adjusted residual score can capture that curve directly from the data instead of me hand-tuning cutoffs per tier.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

The cell below loads `data/raw/content_refresh_anonymized.csv` (30,000 rows, 32 clients, trailing-90-day metrics — see `docs/data-dictionary.md`) and checks three things:

1. CTR clearly falls as position tier worsens (so "position-adjusted" scoring is meaningful, not arbitrary).
2. A large pool of pages already meets the lane guide's own `low_ctr_visible_page` rule (enough impressions, page-1-or-better-ish position, low CTR) — so there's a real backlog to rank, not a handful of edge cases.
3. Even *within* the best tier (`page_1`), there's meaningful spread below the tier's own median — some page-1 pages are underperforming their peers, which is exactly the signal a position-adjusted score is meant to catch.


In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"rows: {len(df):,}  |  clients: {df['client_id'].nunique()}")

# 1) CTR by position tier -- confirms the position-CTR relationship is real and monotonic-ish
ctr_by_tier = (
    df.groupby("position_tier")["ctr"]
    .agg(mean_ctr="mean", median_ctr="median", n="count")
    .sort_values("mean_ctr", ascending=False)
)
print("\nCTR (%) by position tier:")
print(ctr_by_tier)

# 2) size of the low-CTR, visible-page backlog (mirrors the lane guide's low_ctr_visible_page rule)
low_ctr_visible = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
]
print(f"\nlow_ctr_visible_page candidates: {len(low_ctr_visible):,} of {len(df):,} rows "
      f"({len(low_ctr_visible)/len(df):.1%})")

# 3) within-tier spread: page_1 pages sitting well below their own tier's median CTR, with real volume
page1 = df[df["position_tier"] == "page_1"]
tier_median = page1["ctr"].median()
underperformers = page1[(page1["ctr"] < tier_median * 0.5) & (page1["impressions_90d"] >= 500)]
print(f"\npage_1 tier median CTR: {tier_median:.2f}%")
print(f"page_1 pages below half that median with impressions_90d>=500: "
      f"{len(underperformers):,} of {len(page1):,} page_1 rows ({len(underperformers)/len(page1):.1%})")


rows: 30,000  |  clients: 32

CTR (%) by position tier:
               mean_ctr  median_ctr      n
position_tier                             
top_3          1.483611        0.00   2321
page_1         0.652467        0.16  11814
striking       0.323239        0.11   7304
page_3_5       0.222484        0.03   7242
deep           0.150212        0.00   1319

low_ctr_visible_page candidates: 9,759 of 30,000 rows (32.5%)

page_1 tier median CTR: 0.16%
page_1 pages below half that median with impressions_90d>=500: 1,072 of 11,814 page_1 rows (9.1%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work can claim, once built:**
- *Observed:* which pages, on this trailing-90-day snapshot, sit furthest below the expected CTR for their position tier and content type.
- *Directional / decision-support:* a ranked review queue — "review these pages first for a CTR-focused fix" — with reason codes an editor can inspect and disagree with.
- *Comparative:* how a tier-adjusted score or simple model compares to the lane guide's own rule-based baseline (e.g. precision@K on a defined label), evaluated with a client-holdout split so I'm not just re-describing clients the method already saw.

**What this work can never claim:**
- That fixing a title/meta *caused* a CTR increase — I only have observational trailing data, not an experiment (no A/B test, no before/after on the same page post-fix). Any "this fix worked" statement needs a real experiment, not this dataset.
- That anything here reflects how Google's ranking algorithm actually works — I'm only working with FlyRank's observed search/analytics signals, not Google's internals.
- That a low CTR always means a bad title — it could just as easily be a branded query, a SERP feature stealing clicks, or a seasonal dip (per the lane guide's decline/consolidation/seasonality/noise warning), so any "opportunity" label needs a minimum-volume filter and honest confidence language, not a hard yes/no.
- That `is_declining_label` or `trend_direction`/`trend_pct` can be used as *features* — they're derived from the same signal I'd be trying to predict, so using them would just teach a model to copy a rule I already know, not discover anything new.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.